<a href="https://colab.research.google.com/github/Samgoldwin/Image-captioning-model/blob/main/IMAGE_CAPTION_GENERATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Captioning with VGG16 and LSTM

This notebook implements an image captioning model using a combination of a pre-trained VGG16 model for feature extraction and an LSTM-based recurrent neural network for generating descriptive captions. The model is trained and evaluated on the Flickr8k dataset.

**1. Importing Libraries**

In [ ]:
import tensorflow as tf
import re
from pickle import load,dump
import os
from os import listdir
from numpy import array,argmax
from IPython.display import Image,display


from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
from tensorflow.keras.preprocessing.image import load_img,img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout, add
from tensorflow.keras.callbacks import ModelCheckpoint

from nltk.translate.bleu_score import corpus_bleu

**2. Downloading Datasets**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("youssefaboelnasr/flickr8k-text")

Using Colab cache for faster access to the 'flickr8k-text' dataset.


In [ ]:
all_captions = os.path.join(path, "Flickr8k.lemma.token.txt")
test_captions = os.path.join(path, "Flickr_8k.testImages.txt")
train_captions = os.path.join(path, "Flickr_8k.trainImages.txt")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'flickr8k' dataset.
Path to dataset files: /kaggle/input/flickr8k


In [ ]:
import shutil
import os

source_path_images = path # Assuming 'path' still holds the location of the flickr8k images
destination_dir_images = '/content/sample_data/2/'

# Ensure the parent destination directory exists
os.makedirs(destination_dir_images, exist_ok=True)

# Construct the full destination path, keeping the dataset folder name
destination_path_images = os.path.join(destination_dir_images, os.path.basename(source_path_images))

# If the destination directory already exists, remove it to prevent errors from shutil.copytree
if os.path.exists(destination_path_images):
    shutil.rmtree(destination_path_images)

# Copy the directory from the read-only source to the destination
shutil.copytree(source_path_images, destination_path_images)

print(f"Image dataset copied from '{source_path_images}' to '{destination_path_images}'")

# Update the 'path' variable to track the new location
path = destination_path_images

Image dataset copied from '/kaggle/input/flickr8k' to '/content/sample_data/2/flickr8k'


In [ ]:
all_images= "/content/sample_data/2/flickr8k/Images"

In [ ]:
listdir(all_images)

['2045562030_654ddea5e5.jpg',
 '3099923914_fd450f6d51.jpg',
 '3621647714_fc67ab2617.jpg',
 '513269597_c38308feaf.jpg',
 '3471463779_64084b686c.jpg',
 '444481722_690d0cadcf.jpg',
 '3117336911_a729f42869.jpg',
 '3073579130_7c95d16a7f.jpg',
 '3175446111_681a89f873.jpg',
 '3376809186_4e26d880b7.jpg',
 '3272541970_ac0f1de274.jpg',
 '2751466788_4fab701cc3.jpg',
 '2902844125_4186bf3ab6.jpg',
 '2623930900_b9df917b82.jpg',
 '1388373425_3c72b56639.jpg',
 '3349307529_c1a516b9dc.jpg',
 '388386075_9ac3a89ada.jpg',
 '2657484970_610e18144f.jpg',
 '2858903676_6278f07ee3.jpg',
 '2326730558_75c20e5033.jpg',
 '2644430445_47c985a2ee.jpg',
 '2149968397_a7411729d1.jpg',
 '2912476706_9a0dbd3a67.jpg',
 '3211210739_3dea005fde.jpg',
 '272156850_c4445a53f4.jpg',
 '2319808437_bbbdc317c0.jpg',
 '1470536919_1f3fd6c65a.jpg',
 '1859941832_7faf6e5fa9.jpg',
 '3562169000_6aa7f1043d.jpg',
 '2647062476_5ef31ba867.jpg',
 '3027850131_a7772e0ba0.jpg',
 '3356901257_83811a19eb.jpg',
 '2257294002_0073263c54.jpg',
 '3247500085_c

**3. Preprocessing Data**

In [ ]:
file1=open(all_captions,"r")
data1=file1.read()
file1.close()

In [ ]:
data1

'1305564994_00513f9a5b.jpg#0\tA man in street racer armor be examine the tire of another racer \'s motorbike .\n1305564994_00513f9a5b.jpg#1\tTwo racer drive a white bike down a road .\n1305564994_00513f9a5b.jpg#2\tTwo motorist be ride along on their vehicle that be oddly design and color .\n1305564994_00513f9a5b.jpg#3\tTwo person be in a small race car drive by a green hill .\n1305564994_00513f9a5b.jpg#4\tTwo person in race uniform in a street car .\n1351764581_4d4fb1b40f.jpg#0\tA firefighter extinguish a fire under the hood of a car .\n1351764581_4d4fb1b40f.jpg#1\ta fireman spray water into the hood of small white car on a jack\n1351764581_4d4fb1b40f.jpg#2\tA fireman spray inside the open hood of small white car , on a jack .\n1351764581_4d4fb1b40f.jpg#3\tA fireman use a firehose on a car engine that be up on a carjack .\n1351764581_4d4fb1b40f.jpg#4\tFirefighter use water to extinguish a car that be on fire .\n1358089136_976e3d2e30.jpg#0\tA boy sand surf down a hill\n1358089136_976e3d

In [ ]:
descr = dict()
for line in data1.split("\n"):
  token = line.split("\t")
  if len(line) < 2:
    continue
  image_id, image_desc = token[0], token[1]
  image_id = image_id.split(".")[0]
  image_desc = "<start> "+"".join(image_desc)+ "<end>"

  if image_id not in descr:
    descr[image_id] = list()
  descr[image_id].append(image_desc)

In [ ]:
descr['1000268201_693b08cb0e']

['<start> A child in a pink dress be climb up a set of stair in an entry way .<end>',
 '<start> A girl go into a wooden building .<end>',
 '<start> A little girl climb into a wooden playhouse .<end>',
 '<start> A little girl climb the stair to her playhouse .<end>',
 '<start> A little girl in a pink dress go into a wooden cabin .<end>']

In [ ]:
for key,des in descr.items():
  for i in range(len(des)):
    d=des[i].lower()
    d = re.sub(r"\w*[^a-z\s<>]+\w*", "", d)
    des[i] = d
    descr[key] = des

In [ ]:
vocab = set()
for k,v in descr.items():
  [vocab.update(l.split()) for l in v]
print(f"the orginal size of the vocabulary is {len(vocab)}")

the orginal size of the vocabulary is 7079


In [ ]:
vocab

{'traveler',
 'outside<end>',
 'among',
 'hat<end>',
 'twelve',
 'cluster',
 'backlit',
 'adhd',
 'head',
 'internet',
 'fence',
 'flowery',
 'roof',
 'hula',
 'seem',
 'clip',
 'shred',
 'vigorous',
 'fade',
 'reptile',
 'messy',
 'placemats',
 'romantically',
 'couch',
 'unhappy',
 'chow',
 'competiting',
 'ornate',
 'cows',
 'typical',
 'vertical',
 'foil',
 'portrait',
 'bundle',
 'otuside',
 'ruined',
 'bmx',
 'slant',
 'rapids',
 'numbered',
 'smirk',
 'lakefront',
 'carring',
 'dead',
 'pail',
 'nylon',
 'whil',
 'director',
 'bodyboard',
 'welcome',
 'wicket',
 'sandal<end>',
 'untangles',
 'mambo',
 'sunny',
 'expose',
 'plat',
 'flute<end>',
 'mouth<end>',
 'to',
 'anticipate',
 'watery',
 'quintet',
 'pinch',
 'muzzled',
 'shoelace',
 'palestinian',
 'cowgirl',
 'farm',
 'nearby<end>',
 'character',
 'caged',
 'lime',
 'windy',
 'decoration',
 'calm',
 'wan',
 'antiquated',
 'monitor',
 'streched',
 'strip',
 'grass<end>',
 'urban',
 'least',
 'pasta<end>',
 'playful',
 'mor

In [ ]:
def load_text(descr,filename):
  length = 0
  des = dict()
  with open(filename,"r") as file:
    keys = file.read()
    for k in keys.split("\n"):
      if len(k)<2:
        continue
      image_id = k.split(".")[0]
      des[image_id]=descr.get(image_id) # Corrected bug: using image_id instead of undefined 'image'
      length = length +1
  print(f"Number of image : {length}")
  return des

In [ ]:
train_descr  = load_text(descr,train_captions)
test_descr  = load_text(descr,test_captions)

Number of image : 6000
Number of image : 1000


In [ ]:
train_descr["3675825945_96b2916959"]

['<start> a man in skateboard in an empty swim pool <end>',
 '<start> a man ride a skateboard up the side of an empty pool <end>',
 '<start> a skateboarder dress in white be drop into a blue bowl <end>',
 '<start> a skateboarder hang off the edge of a bowl at a skate park <end>',
 '<start> a skateboarder in an empty pool <end>']

### **4. Image Data Preprocessing**

In [ ]:
all_images

'/content/sample_data/2/flickr8k/Images'

In [ ]:
listdir(all_images)

['2045562030_654ddea5e5.jpg',
 '3099923914_fd450f6d51.jpg',
 '3621647714_fc67ab2617.jpg',
 '513269597_c38308feaf.jpg',
 '3471463779_64084b686c.jpg',
 '444481722_690d0cadcf.jpg',
 '3117336911_a729f42869.jpg',
 '3073579130_7c95d16a7f.jpg',
 '3175446111_681a89f873.jpg',
 '3376809186_4e26d880b7.jpg',
 '3272541970_ac0f1de274.jpg',
 '2751466788_4fab701cc3.jpg',
 '2902844125_4186bf3ab6.jpg',
 '2623930900_b9df917b82.jpg',
 '1388373425_3c72b56639.jpg',
 '3349307529_c1a516b9dc.jpg',
 '388386075_9ac3a89ada.jpg',
 '2657484970_610e18144f.jpg',
 '2858903676_6278f07ee3.jpg',
 '2326730558_75c20e5033.jpg',
 '2644430445_47c985a2ee.jpg',
 '2149968397_a7411729d1.jpg',
 '2912476706_9a0dbd3a67.jpg',
 '3211210739_3dea005fde.jpg',
 '272156850_c4445a53f4.jpg',
 '2319808437_bbbdc317c0.jpg',
 '1470536919_1f3fd6c65a.jpg',
 '1859941832_7faf6e5fa9.jpg',
 '3562169000_6aa7f1043d.jpg',
 '2647062476_5ef31ba867.jpg',
 '3027850131_a7772e0ba0.jpg',
 '3356901257_83811a19eb.jpg',
 '2257294002_0073263c54.jpg',
 '3247500085_c

In [ ]:
model = VGG16()
model.layers.pop()
model = Model(inputs=model.inputs, outputs=model.layers[-1].output)
features = dict()
for name in listdir(all_images):
  filename = os.path.join(all_images, name) # Use os.path.join for robust path construction
  if not os.path.isfile(filename): # Check if it's a file
      print(f"Skipping {filename} as it is not a file.")
      continue
  try:
    image = load_img(filename, target_size=(224, 224))
    image = img_to_array(image)
    image=image.reshape((1,image.shape[0],image.shape[1],image.shape[2] ))
    image = preprocess_input(image)
    feature = model.predict(image,verbose=0)
    image_id = name.split(".")[0]
    features[image_id]=feature
  except (UnidentifiedImageError, FileNotFoundError):
    print(f"Could not identify or find image file: {filename}. Skipping.")
    continue

print(f"The number of extracted features: {len(features)}")
dump(features, open("features.dump","wb"))

KeyboardInterrupt: 

In [ ]:
def load_img_feat(fn_features,fn_data):
  length = 0
  feat =  dict()
  file1= open(fn_data,"r")
  file2= open(fn_features,"rb")
  f = load(file2)
  keys = file1.read()
  for k in keys.split("\n"):
    if len(k)<2:
      continue
    image_id = k.split(".")[0] # Corrected line: use 'k' instead of 'image_id'
    feat[image_id] = f[image_id]
    length +=1
  file1.close()
  file2.close()
  print(f"No of Instances {length} ")
  return feat

In [ ]:
train_features=load_img_feat("/content/features.dump",train_captions)
test_features=load_img_feat("/content/features.dump",test_captions)

In [ ]:
test_features["2943334864_6bab479a3e.jpg"]

Text Tokenization:

In [ ]:
def tokenizer(descr):
  d = []
  for key in descr.keys():
    for i in descr[key]:
      d.append(i)
  tokenize = Tokenizer()
  tokenize.fit_on_texts(d)
  return tokenize

In [ ]:
tokenize = tokenizer(train_descr)
vocab_size = len(tokenize.word_index)+1

In [ ]:
print(vocab_size)

5463


In [2]:
d = []
for key in train_descr.keys():
    for i in train_descr[key]:
        d.append(i)

maxlen = max(len(i.split()) for i in d)
print(f"max length if caption in training data is {maxlen}")

NameError: name 'train_descr' is not defined

In [ ]:
def create_sequences(tokenizer,max_length,desc_list,photo,vocab_size):
  x1,x2,y = list(),list(),list()
  for desc in desc_list:
    seq = tokenizer.texts_to_sequences([desc])[0]
    for i in range(1,len(seq)):
      in_seq,out_seq = seq[:i],seq[i]
      in_seq = pad_sequences([in_seq],maxlen=max_length)[0]

      out_seq = to_categorical([out_seq],num_classes = vocab_size)[0]
      x1.append(photo) # Corrected: using 'photo' argument instead of global 'image'
      x2.append(in_seq)
      y.append(out_seq)
  return array(x1),array(x2),array(y)

In [ ]:
def data_generator(descriptions, images, tokenizer, max_length, vocab_size):
    while True: # Changed from 'while 1' to 'while True' for better readability
        for key, desc_list in descriptions.items():
            image_features = images[key][0]
            in_img, in_seq, out_word = create_sequences(tokenizer, max_length, desc_list, image_features, vocab_size)

            # Ensure that actual sequences were generated before yielding
            if len(in_img) > 0:
                yield (in_img, in_seq), out_word

**5. Modelling**

In [ ]:
def define_model(vocab_size, max_length):
    inputs1 = Input(shape=(1000,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation="relu")(fe1)

    # Sequence model
    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, 256)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation="relu")(decoder1)
    outputs = Dense(vocab_size, activation="softmax")(decoder2)

    model = Model(inputs = [inputs1, inputs2], outputs=outputs)

    model.compile(loss="categorical_crossentropy", optimizer="adam")
    print(model.summary())
    return model

**6. Training The Data**

In [ ]:
#Train the model
model = define_model(vocab_size, maxlen)
epochs = 10
steps = len(train_descr)
for i in range(epochs):
    generator = data_generator(train_descr, train_features, tokenize, maxlen, vocab_size)

    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)
    model.save("model_" + str(i) + ".h5")


In [ ]:
def word_for_id(integer,tokenizer):
  for word,index in tokenizer.word_index.items():
    if index == integer:
      return word
  return None

In [ ]:
#generate description for an image
def generate_desc(model, tokenizer, photo, max_length):
    print(f"Type of 'tokenizer' inside generate_desc: {type(tokenizer)}")
    in_text = "<start>"
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        y_hat = model.predict([photo, sequence], verbose=0)
        y_hat = argmax(y_hat)
        word = word_for_id(y_hat, tokenizer)
        if word is None:
            break
        in_text += " " + word
        if word == "end":
            break
    return in_text

**7. Evaluation**

In [ ]:
def evaluate_model(model, descriptions, photos, tokenizer, max_length):
    actual, predicted = list(), list()
    for key, desc_list in descriptions.items():
        yhat = generate_desc(model, tokenizer, photos[key], max_length)
        references = [d.split() for d in desc_list]
        actual.append(references)
        predicted.append(yhat.split())
    print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0,0,0,0)))


In [ ]:
evaluate_model(model,test_descr,test_features,tokenize,maxlen)

In [ ]:
display(Image("/content/sample_data/2/flickr8k/Images/3385593926_d3e9c21170.jpg"))

In [ ]:
yhat=generate_desc(model,tokenize,test_features["3385593926_d3e9c21170"],maxlen)
print(yhat)

In [3]:
max_length = maxlen

NameError: name 'maxlen' is not defined

In [ ]:
dump(tokenize, open("tokenizer.pkl","wb"))

In [4]:
dump(tokenize, open("tokenizer.pkl","wb"))

# Create a separate VGG16 instance for saving its weights/structure for feature extraction
vgg_feature_extractor = VGG16()
vgg_feature_extractor.layers.pop()
vgg_feature_extractor = Model(inputs = vgg_feature_extractor.inputs , outputs = vgg_feature_extractor.layers[-1].output)
# It appears there was an attempt to save the VGG16 model, but the filename 'preprocess.h5' was used,
# while the loading cell 'CwolUsAEN4Y9' expects 'vgg.save'.
# To ensure consistency and enable the 'upload_and_caption_image' function to load the VGG model correctly,
# I will save the VGG feature extractor with the filename 'vgg.save'.
dump(vgg_feature_extractor, open("vgg.save", "wb"))

dump(max_length, open("maxlength.dump","wb"))

NameError: name 'dump' is not defined

**8. Testing The Model**

In [ ]:
from google.colab import files
from tensorflow.keras.models import load_model
from PIL import Image as PILImage
import io
from pickle import load, dump
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.image import img_to_array, preprocess_input

# Load the tokenizer, VGG16 model, and max_length
tokenize = load(open('tokenizer.pkl', 'rb'))
print(f"Type of 'tokenize' after loading in CwolUsAEN4Y9: {type(tokenize)}")
vgg_model = load(open('vgg.save', 'rb'))
max_length = load(open('maxlength.dump', 'rb'))

# Load the trained captioning model
captioning_model = load_model('model_9.h5')

def upload_and_caption_image():
    uploaded = files.upload()

    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')

        # Read the image from bytes
        img_bytes = uploaded[fn]
        img = PILImage.open(io.BytesIO(img_bytes))

        # Preprocess the image
        image = img.resize((224, 224))
        image = img_to_array(image)
        image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
        image = preprocess_input(image)

        # Extract features using the VGG16 model
        feature = vgg_model.predict(image, verbose=0)

        # Generate caption using the loaded captioning model
        caption = generate_desc(captioning_model, tokenize, feature, max_length)

        print(f"Generated Caption: {caption}")

# You can call the function like this to try it out:
# upload_and_caption_image()

In [ ]:
upload_and_caption_image()